# 🚀 LightLLM - Google Colab Training Notebook
Train your custom 124M LightLLM Transformer model on Google Colab GPUs (T4 / A100)!

In [ ]:
# Step 1: Check GPU availability
!nvidia-smi

In [ ]:
# Step 2: Clone repository & install dependencies
!git clone https://github.com/RABNEER/LightLLM.git
%cd LightLLM
!pip install torch numpy tiktoken tqdm

In [ ]:
# Step 3: Prepare dataset (tokenize & binary map)
!python prepare_data.py

In [ ]:
# Step 4: Train LightLLM on Colab GPU
!python train.py

In [ ]:
# Step 5: Test Chat Inference
import torch
from lightllm.model import LightLLM
from lightllm.config import LightLLMConfig
from lightllm.tokenizer import Tokenizer

config = LightLLMConfig()
model = LightLLM(config)
tokenizer = Tokenizer()
checkpoint = torch.load('out/checkpoint.pt', map_location='cuda')
model.load_state_dict(checkpoint['model'], strict=False)
model.to('cuda').eval()

def generate_answer(prompt):
    formatted = f"User: {prompt}\nAssistant:"
    ids = torch.tensor([tokenizer.encode(formatted)], dtype=torch.long).to('cuda')
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=40, temperature=0.2, top_k=5)
    return tokenizer.decode(out[0].tolist()).split('<|endoftext|>')[0]

print("hello ->", generate_answer("hello"))
print("2+2 ->", generate_answer("2+2"))
print("what is your name ->", generate_answer("what is your name"))